# Entrenamiento

LightGBM sobre `data/processed/features.parquet` (25 features, una fila por merchant).

El modelo ordena a los merchants activos: ¿cuáles se parecen a los que una revisión termina suspendiendo?

**Validación (punto 2 de `docs/decisions.md`):** un solo corte, estratificado, 20% de los merchants en test. Semilla 0. `MEA1EE5` ya no está en el dataset.

**Early stopping:** otro 20% del entrenamiento, también estratificado, para decidir cuándo parar. El test no se usa para eso.

**Desbalance:** `scale_pos_weight` = negativos / positivos, con tope en 100.

**Métricas** en el test, por merchant:
- **PR-AUC**, junto a la prevalencia (el azar vale la prevalencia).
- **Recall** en el top 1% y 5% del score.

Con 12 positivos (`MEA1EE5` queda fuera: nunca operó), el test tiene unos 2 o 3. El número es ruidoso.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent
FEATURES_PATH = ROOT / "data" / "processed" / "features.parquet"
OUT_DIR = ROOT / "docs" / "modelo"

SEED = 0
TEST_SIZE = 0.2
ES_SIZE = 0.2
ES_ROUNDS = 50
MAX_POS_WEIGHT = 100
DROP_GROUP_D = False  # MEA1 ya está fuera del dataset; el flag no saca filas

data = pd.read_parquet(FEATURES_PATH)
if DROP_GROUP_D:
    data = data.loc[~data["group_d"]].reset_index(drop=True)
feature_cols = [c for c in data.columns if c not in ("merchant_id", "observation_date", "target", "group_d")]
X = data[feature_cols].astype("float32")
y = data["target"].to_numpy()
merchants = data["merchant_id"].to_numpy()
print(f"{len(data):,} filas, {len(feature_cols)} features, {int(y.sum())} positivos")

## Un corte estratificado

In [ ]:
def new_model(pos_weight):
    return LGBMClassifier(
        n_estimators=2000, learning_rate=0.05, max_depth=3, num_leaves=8,
        min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        reg_lambda=1.0, scale_pos_weight=pos_weight, random_state=SEED, n_jobs=-1, verbose=-1,
    )


idx = np.arange(len(data))
train_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, stratify=y, random_state=SEED)
fit_idx, es_idx = train_test_split(train_idx, test_size=ES_SIZE, stratify=y[train_idx], random_state=SEED)
assert not set(merchants[fit_idx]) & set(merchants[es_idx])
assert not set(merchants[train_idx]) & set(merchants[test_idx])

pos = int(y[fit_idx].sum())
model = new_model(min((len(fit_idx) - pos) / max(pos, 1), MAX_POS_WEIGHT))
model.fit(
    X.iloc[fit_idx], y[fit_idx],
    eval_X=X.iloc[es_idx], eval_y=y[es_idx], eval_metric="average_precision",
    callbacks=[early_stopping(ES_ROUNDS, verbose=False), log_evaluation(0)],
)
y_test = y[test_idx]
score = model.predict_proba(X.iloc[test_idx])[:, 1]
print(f"fit {len(fit_idx):,} ({pos} positivos), early stopping {len(es_idx):,} ({int(y[es_idx].sum())} positivos), "
      f"test {len(test_idx):,} ({int(y_test.sum())} positivos), árboles {model.best_iteration_}")
print("positivos en test:", ", ".join(merchants[test_idx][y_test == 1]))

## Métricas

In [ ]:
def recall_at(frac):
    k = int(np.ceil(frac * len(y_test)))
    top = np.argsort(-score, kind="stable")[:k]
    return y_test[top].sum() / y_test.sum()


metrics = pd.Series({
    "prevalencia": y_test.mean(),
    "PR-AUC": average_precision_score(y_test, score),
    "recall top 1%": recall_at(0.01),
    "recall top 5%": recall_at(0.05),
})
print(metrics.round(4).to_string())

test_merchants = pd.Series(merchants[test_idx], name="merchant_id")
ranking = pd.Series(score, index=test_merchants.values).rank(ascending=False, pct=True)
print("\nPercentil de cada positivo del test (0 = mayor score):")
print(ranking[y_test == 1].sort_values().round(3).to_string())

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
importance = pd.Series(model.booster_.feature_importance("gain"), index=feature_cols)
importance = (importance / importance.sum()).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
precision, recall, _ = precision_recall_curve(y_test, score)
ax[0].plot(recall, precision, color="#378ADD")
ax[0].axhline(y_test.mean(), color="#D85A30", ls="--", label=f"azar = prevalencia ({y_test.mean():.4%})")
ax[0].set_xlabel("recall")
ax[0].set_ylabel("precisión")
ax[0].set_title(f"Curva precisión-recall en test (PR-AUC = {metrics['PR-AUC']:.4f})")
ax[0].legend()
top = importance.head(15)[::-1]
ax[1].barh(top.index, top.values, color="#378ADD")
ax[1].set_xlabel("importancia (ganancia)")
ax[1].set_title("Las 15 features más importantes")
ax[1].tick_params(axis="y", labelsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "evaluacion.png", dpi=110)
plt.show()
metrics.round(5).to_csv(OUT_DIR / "metricas.csv", header=["valor"])
importance.round(5).to_csv(OUT_DIR / "importancia.csv", header=["ganancia"])